# ViT classifier — walkthrough

This notebook runs the happy path end to end against the real trained checkpoint. It imports the actual model class and config from the `vit-classifier` service rather than reimplementing anything inline — a second, silently diverging copy of the pipeline is exactly the class of bug this portfolio has hit before.


In [ ]:
import os

# This notebook lives at notebooks/vit-classifier/; the code it exercises lives at
# services/vit-classifier/. chdir there so the relative paths in Config
# ("checkpoints/...", "data") resolve exactly as they do for `make serve` and the
# trainers, which run from the service directory. Every workspace member is installed
# by `make install`, so no import-path manipulation is needed.
os.chdir(os.path.join("..", "..", "services", "vit-classifier"))

import torch
from ai_residency_common import get_device, load_checkpoint

from config import Config

config = Config()
device = get_device()

print(f"Using device: {device}")
print(f"Classifier arch: {config.classifier_arch}")
print(f"Seq2seq arch:    {config.seq2seq_arch}")
print(f"Classifier checkpoint: {config.classifier_checkpoint_path}")
print(f"Seq2seq checkpoint:    {config.seq2seq_checkpoint_path}")

## 1. Single-digit classifier

Load the real `VisionTransformer` class, restore the committed checkpoint (28x28 input, patch
size 4, `d_model=128`), and run it on real MNIST test images loaded through the same
`MNISTDataset` wrapper the trainer itself uses.


In [ ]:
from vit_classifier.classifier import VisionTransformer

classifier = VisionTransformer(config.classifier_arch)
load_checkpoint(config.classifier_checkpoint_path, classifier)
classifier.to(device).eval()

total_params = sum(p.numel() for p in classifier.parameters())
print(f"Loaded classifier from {config.classifier_checkpoint_path} ({total_params:,} params)")

In [ ]:
from datasets.mnist_dataset import MNISTDataset

# Same wrapper the classifier trainer uses, same defaults. MNIST is already cached under data/.
mnist = MNISTDataset(
    data_dir=config.data_dir,
    image_size=config.classifier_arch.img_size,
    batch_size=8,
    num_workers=0,
)
_, _, test_loader = mnist.get_data_loaders()

images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    logits = classifier(images)
    probs = torch.softmax(logits, dim=-1)
    preds = torch.argmax(logits, dim=-1)
    confidences = torch.max(probs, dim=-1).values

for i in range(len(labels)):
    print(
        f"true={labels[i].item()}  pred={preds[i].item()}  confidence={confidences[i].item():.3f}"
    )

accuracy = (preds == labels).float().mean().item()
print(f"\nBatch accuracy: {accuracy:.2%} ({(preds == labels).sum().item()}/{len(labels)})")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(labels), figsize=(2 * len(labels), 2.2))
for i, ax in enumerate(axes):
    ax.imshow(images[i, 0].cpu(), cmap="gray")
    ax.set_title(f"true {labels[i].item()} / pred {preds[i].item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. (Optional) live API check

If `make serve` is already running on `:8000`, this cell sends one of the real MNIST test images
from Part 1 through `/predict/both` and compares the API's answer to the in-process prediction
above. If the server isn't running the cell says so instead of failing the notebook.


In [ ]:
import base64
import io

import requests
from PIL import Image

try:
    health = requests.get("http://localhost:8000/", timeout=2).json()
    print("Server is up:", health)

    # Re-materialise the first classifier test image as a base64 PNG, the same wire format
    # the API's ImageRequest.image_data expects.
    array = ((images[0, 0].cpu().numpy() * 0.3081 + 0.1307) * 255).clip(0, 255).astype("uint8")
    buffer = io.BytesIO()
    Image.fromarray(array, mode="L").save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")

    response = requests.post(
        "http://localhost:8000/predict/both",
        json={"image_data": encoded},
        timeout=30,
    )
    response.raise_for_status()
    print(f"True label: {labels[0].item()}")
    print("API response:", response.json())
except requests.exceptions.ConnectionError:
    print("The API isn't running on :8000 — skipping the live check.")
    print("Start it with `make serve` to exercise this cell.")